# Hospital Readmissions SQL Analysis
## Querying 70,000+ diabetic patient records to identify readmission risk factors
**Tools:** Python, SQLite, Pandas, SQL  
**Dataset:** UCI Diabetes 130-US Hospitals (1999-2008)  
**Goal:** Use SQL window functions, CTEs, and aggregations to uncover patterns in hospital readmissions

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('hospital.db')

# Preview the data
df = pd.read_sql_query("SELECT * FROM readmissions LIMIT 5", conn)
print(df.shape)
df.head()

(5, 11)


,encounter_id,patient_nbr,race,gender,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,readmitted
0,2278392,8222157,Caucasian,Female,5,1,41,0,1,1,0
1,149190,55629189,Caucasian,Female,15,3,59,0,18,9,0
2,64410,86047875,AfricanAmerican,Female,25,2,11,5,13,6,0
3,500364,82442376,Caucasian,Male,35,2,44,1,16,7,0
4,16680,42519267,Caucasian,Male,45,1,51,0,8,5,0


## Query 1: Readmission Rate by Age Group
Using CASE WHEN to bucket ages and aggregate readmission rates.

In [2]:
query1 = """
SELECT 
    CASE 
        WHEN age < 40 THEN 'Under 40'
        WHEN age BETWEEN 40 AND 60 THEN '40-60'
        ELSE 'Over 60'
    END as age_group,
    COUNT(*) as total_patients,
    SUM(readmitted) as readmissions,
    ROUND(AVG(readmitted) * 100, 2) as readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 2) as avg_days
FROM readmissions
GROUP BY age_group
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql_query(query1, conn)

,age_group,total_patients,readmissions,readmission_rate_pct,avg_days
0,Over 60,47659,4607,9.67,4.51
1,40-60,19344,1386,7.17,3.93
2,Under 40,4515,300,6.64,3.46


## Query 2: Top 10 Most Medicated Patients Using Window Functions
Using RANK() to identify highest-risk patients by medication count.

In [3]:
query2 = """
SELECT 
    patient_nbr,
    age,
    num_medications,
    time_in_hospital,
    readmitted,
    RANK() OVER (ORDER BY num_medications DESC) as medication_rank
FROM readmissions
LIMIT 10
"""
pd.read_sql_query(query2, conn)

,patient_nbr,age,num_medications,time_in_hospital,readmitted,medication_rank
0,24189597,65,81,10,1,1
1,25112691,55,79,12,0,2
2,24526629,65,75,10,0,3
3,43503210,65,75,7,0,3
4,25450911,75,74,13,0,5
5,42522309,75,72,13,1,6
6,43515927,65,72,11,1,6
7,23581485,75,70,13,0,8
8,25162767,55,70,11,1,8
9,24686586,65,69,8,0,10


## Query 3: CTE — High Risk Patient Summary
Using a CTE to first identify high-risk patients, then summarize them.

In [4]:
query3 = """
WITH high_risk AS (
    SELECT *
    FROM readmissions
    WHERE num_medications > 20
    AND time_in_hospital > 7
)
SELECT 
    CASE 
        WHEN age < 40 THEN 'Under 40'
        WHEN age BETWEEN 40 AND 60 THEN '40-60'
        ELSE 'Over 60'
    END as age_group,
    COUNT(*) as high_risk_patients,
    ROUND(AVG(readmitted) * 100, 2) as readmission_rate_pct,
    ROUND(AVG(num_medications), 1) as avg_medications
FROM high_risk
GROUP BY age_group
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql_query(query3, conn)

,age_group,high_risk_patients,readmission_rate_pct,avg_medications
0,Under 40,155,14.84,30.1
1,Over 60,3937,11.76,30.0
2,40-60,1309,10.39,30.9


## Query 4: Subquery — Patients Above Average Hospital Stay
Using a subquery to find patients who stayed longer than the average.

In [5]:
query4 = """
SELECT 
    gender,
    COUNT(*) as patients,
    ROUND(AVG(time_in_hospital), 2) as avg_days,
    ROUND(AVG(readmitted) * 100, 2) as readmission_rate_pct
FROM readmissions
WHERE time_in_hospital > (SELECT AVG(time_in_hospital) FROM readmissions)
GROUP BY gender
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql_query(query4, conn)

,gender,patients,avg_days,readmission_rate_pct
0,Female,14264,7.43,10.77
1,Male,11871,7.48,10.72
2,Unknown/Invalid,1,8.00,0.00


## Key Findings
- Patients over 60 have the highest overall readmission rate (9.67%) and longest stays (4.51 days)
- Among high-risk patients (high medications + long stay), under 40s have the highest read

In [6]:
conn.close()
print("Analysis complete!")

Analysis complete!
